# Concept reanalysis from cached representations
Run after `final_experiment.ipynb`; no GPU is required.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pairs = pd.read_csv("random_concept_pairs.csv")
rows = []
frozen_values = {}
frozen_bootstrap = {}

# Independent class draws within each dataset, shared across conditions/models.
rng = np.random.default_rng(15)
def class_draws():
    return np.concatenate([
        rng.choice(np.flatnonzero(pairs.dataset.eq(dataset)), size=(1000, int(pairs.dataset.eq(dataset).sum())), replace=True)
        for dataset in ["deepset", "safeguard"]], axis=1)
injection_draws = class_draws()
benign_draws = class_draws()

def cosine(first, second):
    denominator = np.linalg.norm(first, axis=-1) * np.linalg.norm(second, axis=-1)
    return np.where(denominator > 1e-8,
        (first * second).sum(axis=-1) / np.maximum(denominator, 1e-8), np.nan)

def vectors(frame, draws=None):
    injections = frame.loc[pairs.injection_row_id].to_numpy(dtype=np.float64)
    benign = frame.loc[pairs.benign_row_id].to_numpy(dtype=np.float64)
    if draws is None:
        return injections.mean(axis=0) - benign.mean(axis=0)
    return injections[draws[0]].mean(axis=1) - benign[draws[1]].mean(axis=1)

for method, seed in [("frozen", 15), ("lora", 15), ("lora", 16), ("lora", 17)]:
    with np.load(f"representations_{method}_{seed}.npz", allow_pickle=False) as cache:
        keys = pd.DataFrame({"row_id": cache["variant_row_ids"], "column": cache["variant_columns"]})
        features = cache["variant_embeddings"]
        conditions = {}
        for style in ["clean", "zero_width", "homoglyph", "base64"]:
            mask = keys["column"].str.endswith("_" + style)
            conditions[style] = pd.DataFrame(features[mask], index=keys.loc[mask, "row_id"])
        clean = vectors(conditions["clean"])
        clean_bootstrap = vectors(conditions["clean"], (injection_draws, benign_draws))
        for style, frame in conditions.items():
            changed = vectors(frame)
            changed_bootstrap = vectors(frame, (injection_draws, benign_draws))
            similarity = float(cosine(clean, changed))
            bootstrap = cosine(clean_bootstrap, changed_bootstrap)
            if method == "frozen":
                frozen_values[style] = similarity
                frozen_bootstrap[style] = bootstrap
            lower, upper = np.nanpercentile(bootstrap, [2.5, 97.5])
            difference_bootstrap = bootstrap - frozen_bootstrap[style]
            change_lower, change_upper = np.nanpercentile(difference_bootstrap, [2.5, 97.5])
            rows.append({"value": method, "seed": seed, "style": style, "pairs": len(pairs),
                "R": similarity, "lower": lower, "upper": upper,
                "change_vs_frozen": similarity - frozen_values[style],
                "change_lower": change_lower, "change_upper": change_upper,
                "clean_concept_norm": np.linalg.norm(clean),
                "perturbed_concept_norm": np.linalg.norm(changed),
                "undefined_bootstrap_samples": int(np.isnan(bootstrap).sum())})
results = pd.DataFrame(rows)
results.to_csv("random_concept_robustness.csv", index=False)
display(results[results["style"].ne("clean")])


,value,seed,style,pairs,R,lower,upper,change_vs_frozen,change_lower,change_upper,clean_concept_norm,perturbed_concept_norm,undefined_bootstrap_samples
1,frozen,15,zero_width,20,-0.066544,-0.278939,0.309760,0.000000,0.000000,0.000000,4.257671,2.081187,0
2,frozen,15,homoglyph,20,0.221331,-0.246805,0.537443,0.000000,0.000000,0.000000,4.257671,2.659109,0
3,frozen,15,base64,20,0.029371,-0.320500,0.353428,0.000000,0.000000,0.000000,4.257671,1.381115,0
5,lora,15,zero_width,20,-0.026856,-0.286045,0.249403,0.039688,-0.285367,0.298672,7.217134,2.264600,0
6,lora,15,homoglyph,20,0.091142,-0.229886,0.450936,-0.130188,-0.289894,0.199351,7.217134,2.639695,0
7,lora,15,base64,20,-0.040756,-0.333218,0.222518,-0.070127,-0.307656,0.281397,7.217134,1.407678,0
9,lora,16,zero_width,20,-0.042934,-0.339122,0.260476,0.023610,-0.351157,0.326899,7.997925,2.214679,0
10,lora,16,homoglyph,20,0.160290,-0.219964,0.550676,-0.061041,-0.321790,0.310749,7.997925,2.751149,0
11,lora,16,base64,20,-0.013282,-0.366207,0.274405,-0.042653,-0.373565,0.372698,7.997925,1.433291,0
13,lora,17,zero_width,20,-0.032117,-0.305711,0.252507,0.034427,-0.350385,0.322897,8.885750,2.237663,0
